In [1]:
import os
# Ensure the working directory is the project root
if os.path.basename(os.getcwd()) == 'R':
    os.chdir('..')
print("Working Directory:", os.getcwd())

Working Directory: c:\order\Desktop\Resume_Parser


# Legacy Resume Parser (spaCy v3)
Fixed CPU compatible version with span trimming.

In [2]:
!pip install -U spacy scikit-learn spacy-transformers
!pip install PyMuPDF

In [3]:
import os
# Clone into R/ folder to keep it isolated
if not os.path.exists('R/CV-Parsing-using-Spacy-3'):
    os.system('git clone https://github.com/laxmimerit/CV-Parsing-using-Spacy-3.git R/CV-Parsing-using-Spacy-3')

In [4]:
import json
cv_data = json.load(open('R/CV-Parsing-using-Spacy-3/data/training/train_data.json', 'r'))
print("Loaded", len(cv_data), "training examples")

Loaded 200 training examples


In [5]:
!python -m spacy init fill-config R/CV-Parsing-using-Spacy-3/data/training/base_config.cfg R/config.cfg

✔ Auto-filled config with all values
✔ Saved config
R\config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [6]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm

def trim_entity_spans(data):
    cleaned_data = []
    for text, annot in data:
        entities = annot['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and text[valid_start].isspace():
                valid_start += 1
            while valid_end > valid_start and text[valid_end - 1].isspace():
                valid_end -= 1
            if valid_end > valid_start:
                valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])
    return cleaned_data

cv_data = trim_entity_spans(cv_data)

def get_spacy_doc(file, data):
    nlp = spacy.blank("en")
    db = DocBin()
    for text, annot in tqdm(data):
        doc = nlp.make_doc(text)
        annot = annot['entities']
        ents = []
        entity_indices = []
        for start, end, label in annot:
            skip_entity = False
            for idx in range(start, end):
                if idx in entity_indices:
                    skip_entity = True
                    break
            if skip_entity:
                continue
            entity_indices = entity_indices + list(range(start, end))
            try:
                span = doc.char_span(start, end, label=label, alignment_mode="contract")
            except:
                continue
            if span is None:
                file.write(str([start, end]) + "    " + str(text) + "\n")
            else:
                ents.append(span)
        try:
            doc.ents = ents
            db.add(doc)
        except:
            pass
    return db

In [7]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(cv_data, test_size=0.3)

with open('R/error.txt', 'w', encoding='utf-8') as file:
    db_train = get_spacy_doc(file, train)
    db_train.to_disk('R/train_data.spacy')
    db_test = get_spacy_doc(file, test)
    db_test.to_disk('R/test_data.spacy')

c:\ProgramData\anaconda3\envs\guns\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 60/60 [00:00<00:00, 156.41it/s]


In [8]:
!python -m spacy train R/config.cfg --output R/output --paths.train R/train_data.spacy --paths.dev R/test_data.spacy

ℹ Saving to output directory: R\output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0        1936.68   1611.23    0.18    0.09    3.56    0.00
⚠ Aborting and saving the final best model. Encountered exception:
ValueError("[E024] Could not find an optimal move to supervise the parser.
Usually, this means that the model can't be updated in a way that's valid and
satisfies the correct annotations specified in the GoldParse. For example, are
all labels added to the model? If you're training a named entity recognizer,
also make sure that none of your annotated entity spans have leading or trailing
whitespace or punctuation. You can also use

I0000 00:00:1784570266.858453   18752 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784570269.104572   18752 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\

In [9]:
import fitz
# If model didn't train successfully, uncomment below line to use pre-trained model:
# nlp = spacy.load('R/CV-Parsing-using-Spacy-3/nlp_model')
nlp = spacy.load('R/output/model-best')

pdf_path = 'data/Alice Clark CV.pdf'
text = " "
with fitz.open(pdf_path) as doc:
    for page in doc:
        text = text + str(page.get_text())

text = ' '.join(text.split())
doc = nlp(text)
for ent in doc.ents:
    print(f"{ent.label_.upper():<15}: {ent.text}")

SKILLS         : Alice
SKILLS         : Clark
SKILLS         : AI
SKILLS         : /
SKILLS         : Machine
SKILLS         : Learning
SKILLS         : Delhi
SKILLS         : ,
SKILLS         : India
SKILLS         : Email
SKILLS         : me
SKILLS         : on
SKILLS         : Indeed
SKILLS         : •
SKILLS         : 20
SKILLS         : +
SKILLS         : years
SKILLS         : of
SKILLS         : experience
SKILLS         : in
SKILLS         : data
SKILLS         : handling
SKILLS         : ,
SKILLS         : design
SKILLS         : ,
SKILLS         : and
SKILLS         : development
SKILLS         : •
SKILLS         : Data
SKILLS         : Warehouse
SKILLS         : :
SKILLS         : Data
SKILLS         : analysis
SKILLS         : ,
SKILLS         : star
SKILLS         : /
SKILLS         : snow
SKILLS         : flake
SKILLS         : scema
SKILLS         : data
SKILLS         : modelling
SKILLS         : and
SKILLS         : design
SKILLS         : specific
SKILLS         : to
